In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class BahdAtt(nn.Module):
    def __init__(self,enc_in=10,dec_in=8,hidden=32,out_steps=4,out_dim=3):
        super().__init__()
        self.out_steps = out_steps
        self.enc = nn.LSTM(enc_in,hidden,batch_first=True,bidirectional=True)
        self.dec = nn.LSTM(dec_in,hidden,batch_first=True)
        self.w1 = nn.Linear(hidden * 2,hidden,bias=False)
        self.w2 = nn.Linear(hidden,hidden,bias=False)
        self.v = nn.Linear(hidden,1,bias=False)
        self.bridge_h = nn.Linear(hidden * 2,hidden)
        self.bridge_c = nn.Linear(hidden * 2,hidden)
        self.head = nn.Linear(hidden *2 + hidden,out_dim)
    def forward(self,src,tgt):
        enc_out,(h,c) = self.enc(src)
        h_dec = torch.cat([h[-2],h[-1]],dim=1)
        c_dec = torch.cat([c[-2],c[-1]],dim=1)
        h0 = self.bridge_h(h_dec).unsqueeze(0)
        c0 = self.bridge_c(c_dec).unsqueeze(0)
        dec_out,_ = self.dec(tgt,(h0,c0))
        scores = self.v(torch.tanh(
            self.w1(enc_out).unsqueeze(1) + self.w2(dec_out).unsqueeze(2)
        )).squeeze(-1)
        attn = F.softmax(scores,dim=-1)
        ctx = torch.bmm(attn,enc_out)
        out = self.head(torch.cat([dec_out,ctx],dim=-1))
        return out
model = BahdAtt()
model.eval()
with torch.no_grad():
    out = model(torch.randn(3,12,10),torch.randn(3,4,8))
    print('输出形状：', out.shape if isinstance(out, torch.Tensor) else type(out))       


输出形状： torch.Size([3, 4, 3])


In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class ScalerAtt(nn.Module):
    def __init__(self,d_model=32,n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.n_heads = n_heads
        self.q = nn.Linear(d_model,d_model)
        self.k = nn.Linear(d_model,d_model)
        self.v = nn.Linear(d_model,d_model)
        self.out = nn.Linear(d_model,d_model)
    def forward(self,x):
        B,T,D = x.shape
        print("B,T,D",B,T,D)
        Q = self.q(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        print('Q:',Q.shape)
        K = self.k(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        print('K:',K.shape)
        V = self.v(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        print('V:',V.shape)
        scores = (Q @ K.transpose(-2,-1)) / (self.d_head ** 0.5)
        print('scores:',scores.shape)
        weight = F.softmax(scores,dim=-1)
        print('weight:',weight.shape)
        attended = (weight @ V).transpose(1,2).contiguous().view(B,T,D)
        print('attended:',attended.shape)
        
        return self.out(attended)
model = ScalerAtt()
model.eval()
with torch.no_grad():
    out = model(torch.randn(4,16,32))
    print(out.shape)
    print(model)

B,T,D 4 16 32
Q: torch.Size([4, 4, 16, 8])
K: torch.Size([4, 4, 16, 8])
V: torch.Size([4, 4, 16, 8])
scores: torch.Size([4, 4, 16, 16])
weight: torch.Size([4, 4, 16, 16])
attended: torch.Size([4, 16, 32])
torch.Size([4, 16, 32])
ScalerAtt(
  (q): Linear(in_features=32, out_features=32, bias=True)
  (k): Linear(in_features=32, out_features=32, bias=True)
  (v): Linear(in_features=32, out_features=32, bias=True)
  (out): Linear(in_features=32, out_features=32, bias=True)
)


In [34]:
import torch
import torch.nn as nn
class Inception(nn.Module):
    def __init__(self,in_ch=64):
        super().__init__()
        self.b1 = nn.Conv2d(in_ch,32,kernel_size=1)
        self.b2 = nn.Sequential(
            nn.Conv2d(in_ch,32,1),
            nn.Conv2d(32,64,3,padding=1)
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(in_ch,16,1),
            nn.Conv2d(16,32,5,padding=2)
        )
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3,stride=1,padding=1),
            nn.Conv2d(in_ch,16,1)
        )
    def forward(self,x):
        return torch.cat([self.b1(x),self.b2(x),self.b3(x),self.b4(x)],dim=1)
model = Inception()
model.eval()
with torch.no_grad():
    out = model(torch.randn(2,64,28,28))
    print(out.shape)

torch.Size([2, 144, 28, 28])
